In [ ]:
### K-평균 알고리즘

## 외부 데이터 저장 & 넘파이 형태로 로드
!wget https://bit.ly/fruits_300_data -O fruits_300.npy

import numpy as np
fruits = np.load('fruits_300.npy')
fruits_2d = fruits.reshape(-1, 100*100) # fruits 크기: (300, 100, 100) => (300, 10000)(2차원)



## 사이킷런 k-평균 알고리즘 호출
from sklearn.cluster import KMeans
km = KMeans(n_clusters = 3, random_state = 42)
km.fit(fruits_2d)

print(km.labels_) # 클러스터링 결과 확인
print(np.unique(km.labels_, return_counts = True)) # 각 클러스터에 속한 데이터 개수 확인



## K-평균 분류 결과 시각화 - draw_fruits 함수를 정의하여

import matplotlib.pyplot as plt

def draw_fruits(arr, ratio = 1): # 입력 인수 => 배열
  n = len(arr) # n은 샘플 개수(=배열 길이) 저장

  # .ceil => 정수 올림 메서드; 필요한 행 개수 계산 EX_ 98개 샘플 => ceil(9.8) => 10행 필요
  rows = int(np.ceil(n/10))

  # row가 2보다 작다면 => row = 1 => 샘플 개수가 10개 이하인 상황 => 열 개수 = n
  cols = n if rows < 2 else 10

  # 행과 열 개수만큼 서브 그래프 형성
  fig, axs = plt.subplots(rows, cols, figsize = (cols*ratio, rows*ratio), squeeze = False)

  for i in range(rows):
    for j in range(cols):
      if i*10 + j < n: # n개까지만 그리기 EX_ n=98일 때, i=9, j=7인 경우 모든 샘플 ok => 중단
        axs[i, j].imshow(arr[i*10 + j], cmap = 'gray_r')
        axs[i, j].axis('off')
  plt.show()

draw_fruits(fruits[km.labels_ == 0]) # 클러스터링 결과가 0인 샘플만 그리기
draw_fruits(fruits[km.labels_ == 1]) # 클러스터링 결과가 1인 샘플만 그리기
draw_fruits(fruits[km.labels_ == 2]) # 클러스터링 결과가 2인 샘플만 그리기



## 클러스터 중심

# 클러스터 중심(3개)의 픽셀을 2차원(행과 열) 배열로 변환 => 그리기
draw_fruits(km.cluster_centers_.reshape(-1, 100, 100))

# 훈련 데이터 - 클러스터 중심 거리를 산출하는 transform() 메서드
print(km.transform(fruits_2d[100:101])) # 100번째 훈련 샘플과 클러스터 중심(3개) 거리 산출

print(km.predict(fruits_2d[100:101])) # 100번 샘플: 중심까지 거리가 가장 가까운 0번 클러스터로 분류.

draw_fruits(fruits[100:101]) # 100번 샘플 그리기

print(km.n_iter_) # 반복 횟수



## 최적의 k 찾기 - 이너셔(inertia)와 엘보우(ellbow) 기법

inertia = []

for k in range(2, 7):
  km = KMeans(n_clusters = k, n_init = 'auto', random_state = 42)
  km.fit(fruits_2d)
  inertia.append(km.inertia_)

# 이너셔 감소 폭이 완화되는(느려지는) 구간에서 최적의 k 도출
plt.plot(range(2, 7), inertia)
plt.xlabel('k')
plt.ylabel('intertia')
plt.show()